In [ ]:
import time
import torch
import numpy as np
from typing import Optional, Tuple, List, Dict
from dataclasses import dataclass


@dataclass
class polyVeloxConfig:
    num_sweeps: int = 1000
    num_samples: int = 1024
    beta_start: float = 0.1
    beta_end: float = 5.0
    alpha_start: float = 0.3
    alpha_end: float = 1.0
    annealing_schedule: str = 'linear'
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    verbose: bool = False


class polyVeloxSolver:
    
    def __init__(self, 
                 H1: np.ndarray,           # линейные (N,)
                 H2: np.ndarray,           # квадратичные (N, N)
                 H3: np.ndarray,           # кубические (N, N, N)
                 config: Optional[polyVeloxConfig] = None):
        
        self.config = config or VeloxQHUBOConfig()
        self.device = torch.device(self.config.device)
        
       
        torch.cuda.empty_cache()
        
        self.N = H1.shape[0]
        
        self.H1 = torch.tensor(H1, dtype=torch.float32, device=self.device)
        self.H2 = torch.tensor(H2, dtype=torch.float32, device=self.device)
        self.H3 = torch.tensor(H3, dtype=torch.float32, device=self.device)
        
        
        self.H2_diag = self.H2.diag().clone()
        

    
    def _energy(self, x: torch.Tensor) -> torch.Tensor:

        energy = torch.einsum('ij,bi,bj->b', self.H2, x, x) + torch.einsum('i,bi->b', self.H1, x)
        

        energy += torch.einsum('ijk,bi,bj,bk->b', self.H3, x, x, x)
        
        return energy
    
    def _local_field(self, x: torch.Tensor) -> torch.Tensor:

        field = self.H1 + 2 * torch.matmul(x, self.H2)

        field += 3 * torch.einsum('ijk,bj,bk->bi', self.H3, x, x)
        
        return field
    
    def solve(self) -> Tuple[np.ndarray, float, Dict]:
        
        x = torch.randint(0, 2, (self.config.num_samples, self.N), 
                         device=self.device).float() * 2 - 1
        
        best_energy = float('inf')
        best_x = None
        data_x = []
        
        for sweep in range(self.config.num_sweeps):
            t = sweep / self.config.num_sweeps

            if self.config.annealing_schedule == 'linear':
                beta = self.config.beta_start + t * (self.config.beta_end - self.config.beta_start)
                alpha = self.config.alpha_start + t * (self.config.alpha_end - self.config.alpha_start)
            else:  # exp
                beta = self.config.beta_start * (self.config.beta_end / self.config.beta_start) ** t
                alpha = self.config.alpha_start * (self.config.alpha_end / self.config.alpha_start) ** t
            
            
            field = self._local_field(x)

            delta_E = (1 - 2*x) * field
            accept_prob = torch.exp(-beta * delta_E)
            accept_prob = alpha * torch.clamp(accept_prob, max=1.0)
            
            flip = torch.rand_like(x) < accept_prob
            x = torch.where(flip, -x, x)
            
            # Мониторинг
            if sweep % 20 == 0:
                data_x.append(x.cpu())
                energies = self._energy(x)
                min_energy = energies.min().item()
                
                if min_energy < best_energy:
                    best_energy = min_energy
                    best_idx = energies.argmin().item()
                    best_x = x[best_idx].cpu().numpy().copy()
                
               
                
                if self.config.verbose:
                    print(f"  Sweep {sweep}, E={min_energy:.4f}, E_min={best_energy}, β={beta:.2f}")

        torch.cuda.empty_cache()
        

        
        return best_x.astype(int), data_x




In [ ]:
def compute_energy_sparse(x, H1, H2, H3):
    """
    Эффективное вычисление энергии для разреженных тензоров
    """
    energy = 0.0
    n = len(x)
    
    # Линейные
    energy += np.dot(x, H1)
    
    # Квадратичные (только верхний треугольник)
    for i in range(n):
        for j in range(i+1, n):
            if H2[i, j] != 0:
                energy += H2[i, j] * x[i] * x[j]
    
    # Кубические (только i < j < k)
    for i in range(n):
        for j in range(i+1, n):
            for k in range(j+1, n):
                if H3[i, j, k] != 0:
                    energy += H3[i, j, k] * x[i] * x[j] * x[k]
    
    return energy

In [ ]:
for i in range(0, 20):
    H1=np.zeros(200) 
    H2=np.zeros((200, 200)) 
    H3=np.load(f"J_T_h_1/T_{i}.npy")

    config = polyVeloxConfig(
        num_sweeps=1000,
        num_samples=5000,
        beta_start= 0.01,
        beta_end= 6,
        alpha_start= 0.5,
        alpha_end= 0.6,
        annealing_schedule='exp',
        verbose=False
    )

    solver = polyVeloxSolver(H1/np.std(H3), H2/np.std(H3), H3/np.std(H3), config)

    spins, data_x = solver.solve()
    print(i, compute_energy_sparse(spins, H1, H2, H3))
    np.save(f"solution_data_1/polyVelox_{i}.npy", data_x)